# Model Selection Guide — Tabular Regression

A cheat-sheet for choosing a regression model on tabular data (e.g. price prediction).

## Quick decision guide

| Situation | Recommended model |
|---|---|
| Fast, interpretable baseline | **Ridge / Linear** |
| Mostly numeric, strong model with ~zero tuning | **Random Forest** / **HistGradientBoosting** |
| Lots of missing values, mixed types, best accuracy | **LightGBM** / **XGBoost** / **HistGBR** |
| Many high-cardinality categoricals (brand, model, city) | **CatBoost** (or target-encode + LightGBM) |
| Large dataset (100k+ rows), speed matters | **LightGBM** |
| Squeeze out the last bit of accuracy | **Stacking ensemble** |
| Few rows (< ~1k), simple relationships | **Ridge / Lasso** |

**Rule of thumb:** start with a gradient-boosted tree (LightGBM / XGBoost / HistGBR).
They almost always win on tabular data with mixed types, non-linearity, and missing
values. Use Ridge only as a sanity-check baseline.

## Per-model pros & cons

| Model | NaN native | Categoricals native | Pros | Cons |
|---|:--:|:--:|---|---|
| **Ridge / Linear** | ❌ | ❌ (one-hot) | Fast, interpretable, hard to overfit | Linear only; needs scaling + imputation + one-hot; misses interactions |
| **Random Forest** | ❌ | ❌ | Robust, little tuning, feature importance | Needs imputation/encoding; weaker than GBMs; poor extrapolation |
| **Extra Trees** | ❌ | ❌ | Faster than RF, extra variance reduction | Same limits as RF; slightly more bias |
| **HistGradientBoosting** | ✅ | ✅ (≤255) | Native NaN + cats, fast, no extra deps | Cat cap 255 (target-encode high-card); fewer knobs |
| **XGBoost** | ✅ | partial | Top accuracy, very tunable, GPU | Needs encoding for cats; many params; slower than LGBM |
| **LightGBM** | ✅ | ✅ (category dtype) | Fastest GBM, low memory, top accuracy | Can overfit small data; many params |
| **CatBoost** | ✅ | ✅ (best) | Best native categorical handling, strong defaults | Slower; extra dependency |
| **Stacking** | — | — | Usually most accurate (combines models) | Complex, slow, marginal gains, hard to maintain |

## Models usually NOT ideal here

- **kNN** — suffers in high dimensions, slow predict, needs scaling/imputation.
- **SVR** — scales poorly to large data, sensitive to scaling, hard to tune.
- **Plain Decision Tree** — high variance, overfits; use ensembles instead.
- **Neural nets (MLP)** — need lots of data/tuning to match GBMs on tabular.

## Practical workflow

1. **Baseline:** Ridge (one-hot + scale + impute) — confirms non-linearity exists.
2. **Default strong model:** `HistGradientBoostingRegressor` (NaN + cats native).
3. **Compare GBMs:** LightGBM / XGBoost / CatBoost with target-encoded high-card cols.
4. **Tune the best** with Optuna (see the *Tuning* section below).
5. **Optional:** blend or stack the top 2–3 for a final small gain.

> Evaluate with shuffled 5-fold CV on the **log target**; report a dollar/%
> metric (MAE, MdAPE) alongside RMSE for interpretability.


# Template Codes — Encoding Strategies & Full Model Comparison

Reconstructed from working notebooks. Two reusable blocks:
1. A **manual K-Fold target encoder** with smoothing (leakage-safe).
2. A full **encoding-strategy comparison** template (Label / Target / One-Hot) across tree and linear models, ending in a stacking ensemble.

> Note: assumes `train_df` / `test_df` (or `train_clean`) are already loaded with the relevant columns.


In [ ]:
# --- Target Encoding for Manufacturer and Model ---
# Uses K-Fold to prevent data leakage: each fold's encoding is computed from other folds only

from sklearn.model_selection import KFold


def target_encode(train_df, test_df, col, target_col, n_splits=5, smoothing=10):
    """K-Fold target encoding with smoothing to prevent leakage and overfitting."""
    global_mean = train_df[target_col].mean()

    # For training data: use out-of-fold encoding
    train_encoded = pd.Series(np.nan, index=train_df.index)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(train_df):
        # Compute mean from training fold only
        fold_means = train_df.iloc[train_idx].groupby(col)[target_col].agg(['mean', 'count'])
        # Smoothing: blend category mean with global mean based on count
        smooth_mean = (fold_means['mean'] * fold_means['count'] + global_mean * smoothing) / (fold_means['count'] + smoothing)
        # Apply to validation fold
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][col].map(smooth_mean)

    # Fill any NaN (unseen in fold) with global mean
    train_encoded = train_encoded.fillna(global_mean)

    # For test data: use full training data encoding
    full_means = train_df.groupby(col)[target_col].agg(['mean', 'count'])
    smooth_full = (full_means['mean'] * full_means['count'] + global_mean * smoothing) / (full_means['count'] + smoothing)
    test_encoded = test_df[col].map(smooth_full).fillna(global_mean)

    return train_encoded, test_encoded


# Apply target encoding
train_clean['Manufacturer_target_enc'], test_df['Manufacturer_target_enc'] = target_encode(
    train_clean, test_df, 'Manufacturer', 'Log_Price'
)
train_clean['Model_target_enc'], test_df['Model_target_enc'] = target_encode(
    train_clean, test_df, 'Model', 'Log_Price'
)

print("Target encoding complete:")
print(f"  Manufacturer_target_enc: train mean={train_clean['Manufacturer_target_enc'].mean():.3f}, "
      f"std={train_clean['Manufacturer_target_enc'].std():.3f}")
print(f"  Model_target_enc:        train mean={train_clean['Model_target_enc'].mean():.3f}, "
      f"std={train_clean['Model_target_enc'].std():.3f}")
print(f"\n  Top 5 manufacturers by target encoding:")
print(train_clean.groupby('Manufacturer')['Manufacturer_target_enc'].first().sort_values(ascending=False).head())


# Template: Encoding Strategies & Full Model Comparison

Accuracy ceiling (typical): Stacking > CatBoost ≈ LightGBM > XGBoost > ...


## 1. Label Encoding (for all tree models)

Assigns arbitrary integers. Trees split on thresholds so it works, but provides no information about category-target relationship.


In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, TargetEncoder, OneHotEncoder
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor)
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor


# --- Setup ---
high_card = ['make', 'model', 'city']
low_card = [c for c in cat_cols if c not in high_card]
y = train_df['price_log']
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.drop(['price', 'price_log']).tolist()
print(f"Numeric columns ({len(num_cols)}):")
print(num_cols)
cat_cols = train_df.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}):")
print(cat_cols)


In [ ]:
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined train+test to handle unseen categories
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col + '_enc'] = le.transform(train_df[col].astype(str))
    test_df[col + '_enc'] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

# Feature columns for modeling
feature_cols = num_cols + [c + '_enc' for c in cat_cols]

print(f"\nTotal features for modeling: {len(feature_cols)}")
print(f"\nSample encoded data:")
print(train_df[feature_cols].head())


In [ ]:
# ============================================================
# 1. LABEL ENCODING (all categoricals -> integers)
# ============================================================
# Already computed above as feature_cols with '_enc' suffix
X_label = train_df[feature_cols].copy()
print(f"Label-encoded features shape: {X_label.shape}")
print(f"High-cardinality unique counts: {[(c, train_df[c].nunique()) for c in high_card]}")


## 2. Target Encoding (for high cardinality)

Replaces each category with the mean of the target for that category. Uses internal cross-fitting to prevent leakage. Much more informative for tree models than arbitrary integers.


In [ ]:
# ============================================================
# 2. TARGET ENCODING (high cardinality) + ORDINAL (low cardinality)
# ============================================================
X_target = train_df[num_cols + cat_cols].copy()

# Target encode high-cardinality (leakage-safe via internal CV)
te = TargetEncoder(random_state=42)
X_target[high_card] = te.fit_transform(X_target[high_card].astype(str), y)

# Ordinal encode low-cardinality
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_target[low_card] = oe.fit_transform(X_target[low_card].astype(str))

print(f"Target-encoded features shape: {X_target.shape}")
print(f"\nSample of target-encoded high-card columns:")
print(X_target[high_card].head())


## 3. Compare Label Encoding vs Target Encoding

Run the same model (LightGBM) on both encodings to see the difference.


In [ ]:
# ============================================================
# 3. COMPARE: Label Encoding vs Target Encoding
# ============================================================
lgbm_params = dict(n_estimators=500, max_depth=8, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)

# Label encoded
scores_label = cross_val_score(lgb.LGBMRegressor(**lgbm_params), X_label, y,
                               cv=kf, scoring='neg_root_mean_squared_error')
# Target encoded
scores_target = cross_val_score(lgb.LGBMRegressor(**lgbm_params), X_target, y,
                                cv=kf, scoring='neg_root_mean_squared_error')

print(f"{'Encoding':<20} | {'RMSE':>8} | {'STD':>6}")
print(f"{'-'*20}-+-{'-'*8}-+-{'-'*6}")
print(f"{'Label Encoding':<20} | {-scores_label.mean():.4f} | {scores_label.std():.4f}")
print(f"{'Target Encoding':<20} | {-scores_target.mean():.4f} | {scores_target.std():.4f}")
print(f"\nImprovement: {(-scores_label.mean()) - (-scores_target.mean()):.4f} RMSE")


## 4. Ridge Regression with One-Hot Encoding

One-hot is required for Ridge since it's a linear model — label-encoded integers would imply a false ordinal relationship.


In [ ]:
# ============================================================
# 4. RIDGE WITH ONE-HOT ENCODING (proper linear model baseline)
# ============================================================
X_ridge_raw = train_df[num_cols + cat_cols].copy()

# Pipeline: one-hot encode categoricals, passthrough numerics, then Ridge
cat_indices = [X_ridge_raw.columns.get_loc(c) for c in cat_cols]
num_indices = [X_ridge_raw.columns.get_loc(c) for c in num_cols]

ridge_pipe = Pipeline([
    ('encoder', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols),
        ('num', 'passthrough', num_cols),
    ])),
    ('ridge', Ridge(alpha=1.0))
])

scores_ridge_ohe = cross_val_score(ridge_pipe, X_ridge_raw, y,
                                   cv=kf, scoring='neg_root_mean_squared_error')
print(f"Ridge + One-Hot  RMSE: {-scores_ridge_ohe.mean():.4f} ± {scores_ridge_ohe.std():.4f}")
print(f"Ridge + LabelEnc RMSE: {results['Ridge']['RMSE']:.4f} ± {results['Ridge']['STD']:.4f}")
print(f"\nOne-hot improvement: {results['Ridge']['RMSE'] - (-scores_ridge_ohe.mean()):.4f} RMSE")


## 5. All Models with Best Encoding (Target Encoded)

Run all models on the target-encoded features for a fair comparison.


In [ ]:
# ============================================================
# 5. ALL MODELS — using target-encoded features (X_target)
# ============================================================
all_models = {
    'Ridge (target-enc)': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=500, max_depth=25, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, num_leaves=63,
                                  subsample=0.8, colsample_bytree=0.8,
                                  reg_alpha=0.1, reg_lambda=1.0,
                                  random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=8,
                                  verbose=0, random_state=42),
    'HistGBR': HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, random_state=42),
}

# Evaluate all
print(f"{'Model':<20} | {'RMSE':>8} | {'STD':>6}")
print(f"{'-'*20}-+-{'-'*8}-+-{'-'*6}")

results_te = {}
for name, mdl in all_models.items():
    scores = cross_val_score(mdl, X_target, y, cv=kf, scoring='neg_root_mean_squared_error')
    results_te[name] = {'RMSE': -scores.mean(), 'STD': scores.std()}
    print(f"{name:<20} | {-scores.mean():.4f} | {scores.std():.4f}")

# Also add Ridge with one-hot for reference
results_te['Ridge (one-hot)'] = {'RMSE': -scores_ridge_ohe.mean(), 'STD': scores_ridge_ohe.std()}
print(f"{'Ridge (one-hot)':<20} | {-scores_ridge_ohe.mean():.4f} | {scores_ridge_ohe.std():.4f}")

print(f"\n✓ Best model: {min(results_te, key=lambda k: results_te[k]['RMSE'])}")


In [ ]:
best_model_name = 'XGBoost'
best_model = all_models[best_model_name]
best_model.fit(X_label, y)

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    importance.head(15).plot.barh()
    plt.title(f'{best_model_name} - Top 15 Feature Importances')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Generate predictions and create submission
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)   # Back-transform from log scale
y_pred = np.maximum(y_pred, 0)  # Ensure no negative prices

# Create submission file
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': y_pred.astype(int)})
submission.to_csv('submission_random_forest.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Prediction stats:\n{submission['Price'].describe()}")
print("\nFirst 5 predictions:")
print(submission.head())

## 6. Stacking Ensemble (optional — combine top models)

Combine the best 3 models with Ridge as meta-learner for maximum accuracy.


In [ ]:
# ============================================================
# 6. STACKING ENSEMBLE
# ============================================================
stack = StackingRegressor(
    estimators=[
        ('lgbm', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=42, n_jobs=-1, verbose=-1)),
        ('xgb', xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)),
        ('catboost', CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8,
                                       verbose=0, random_state=42)),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

scores_stack = cross_val_score(stack, X_target, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Stacking (LGB+XGB+CatBoost → Ridge) RMSE: {-scores_stack.mean():.4f} ± {scores_stack.std():.4f}")


# Tuning — Hyperparameter Search (Optuna) for LightGBM & XGBoost

Template adapted from `car_prices.ipynb`. Bayesian search with Optuna over CV RMSE,
then trains the tuned models and blends them.

> **Inputs expected:** a feature matrix `X` and target `y` (log price), already encoded
> (e.g. the `X_target` / `y` from the encoding section above). Set them once below.
> Optuna minimizes 5-fold CV RMSE (`neg_root_mean_squared_error`).


In [ ]:
import optuna
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- point these at your prepared data (target-encoded features + log target) ---
X_train = X_target          # <- e.g. the target-encoded matrix from the encoding section
y_train = y                 # <- log price

kf = KFold(n_splits=5, shuffle=True, random_state=42)
N_TRIALS = 50               # raise for a more thorough search (slower)


In [ ]:
# --- Tune LightGBM ---
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

print(f"Tuning LightGBM ({N_TRIALS} trials)...")
lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest LightGBM RMSE: {lgb_study.best_value:.4f}")
print(f"Best params: {lgb_study.best_params}")


In [ ]:
# --- Tune XGBoost ---
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'random_state': 42, 'n_jobs': -1,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

print(f"Tuning XGBoost ({N_TRIALS} trials)...")
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest XGBoost RMSE: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")


In [ ]:
# --- Train tuned models and blend predictions ---
# (requires X_test prepared with the SAME encoding as X_train)
print("Training tuned LightGBM...")
lgb_best = lgb.LGBMRegressor(**lgb_study.best_params, random_state=42, n_jobs=-1, verbose=-1)
lgb_best.fit(X_train, y_train)

print("Training tuned XGBoost...")
xgb_best = xgb.XGBRegressor(**xgb_study.best_params, random_state=42, n_jobs=-1)
xgb_best.fit(X_train, y_train)

# Check blended CV score (out-of-fold) before trusting the blend
from sklearn.model_selection import cross_val_predict
lgb_cv = cross_val_predict(lgb_best, X_train, y_train, cv=kf)
xgb_cv = cross_val_predict(xgb_best, X_train, y_train, cv=kf)
blended_cv = 0.5 * lgb_cv + 0.5 * xgb_cv
blended_rmse = np.sqrt(mean_squared_error(y_train, blended_cv))

print(f"\nBlended CV RMSE: {blended_rmse:.4f}")
print(f"LightGBM best:   {lgb_study.best_value:.4f}")
print(f"XGBoost best:    {xgb_study.best_value:.4f}")
print(f"Blend improvement: {min(lgb_study.best_value, xgb_study.best_value) - blended_rmse:.4f}")

# Final predictions on test set (uncomment when X_test is ready):
# blended_pred_log = 0.5 * lgb_best.predict(X_test) + 0.5 * xgb_best.predict(X_test)
# blended_pred = np.maximum(np.expm1(blended_pred_log), 0)
